Rolling Date Window Filter
===
Difficulty: Medium

Problem Description:
===================
You work at a bank. Given a `transactions` table, find all users who transacted on **every single
day** within the first 5 days of January 2020. Return the count of such users.

Sample Input:
```
| user_id | created_at |
|---------|------------|
| 1       | 2020-01-01 |
| 1       | 2020-01-02 |
| 1       | 2020-01-03 |
| 1       | 2020-01-04 |
| 1       | 2020-01-05 |
| 2       | 2020-01-01 |
| 2       | 2020-01-03 |  ← missed Jan 2
```

Sample Output:
```
number_of_users: 1   (only user 1 was active all 5 days)
```

In [ ]:
import pandas as pd

transactions = pd.DataFrame({
    'user_id':    [1,1,1,1,1, 2,2,2, 3,3,3,3,3,3, 4,4],
    'created_at': pd.to_datetime([
        # User 1: all 5 days ✅
        '2020-01-01','2020-01-02','2020-01-03','2020-01-04','2020-01-05',
        # User 2: only 3 of 5 ❌
        '2020-01-01','2020-01-03','2020-01-05',
        # User 3: all 5 + extra ✅
        '2020-01-01','2020-01-02','2020-01-03','2020-01-04','2020-01-05','2020-01-05',
        # User 4: outside window ❌
        '2020-01-06','2020-01-07'
    ])
})
print(transactions)

**Concepts to use:**
1. **`between(start, end)`** — boolean filter for date range (inclusive on both ends).
2. **`groupby().nunique()`** — count *distinct* days per user (handles multiple txns same day).
3. **Boolean comparison on Series** — `days_per_user == 5` creates a boolean Series; `.sum()` counts Trues.
4. **`pd.Timedelta`** — alternative: filter with `>= start_date` and `<= start_date + pd.Timedelta(days=4)`.

In [ ]:
# Optimised Solution
def users_active_all_5_days(df):
    # Step 1: filter to window Jan 1-5 2020
    window = df[df['created_at'].between('2020-01-01', '2020-01-05')].copy()

    # Step 2: normalize to date (removes time component if any)
    window['date'] = window['created_at'].dt.date

    # Step 3: count distinct days per user
    days_per_user = window.groupby('user_id')['date'].nunique()

    # Step 4: count users with exactly 5 distinct days
    return int((days_per_user == 5).sum())

print(f"Users active all 5 days: {users_active_all_5_days(transactions)}")